# Suite FRM — The framework

Every dialect implements the same protocols (`mbse.Expressions.Framework`): its expressions are serializable
through meta-schemas and builders, structurally traversable through forms, validatable, evaluatable and
translatable. `Framework.Terms.Declared` derives most of a dialect from its kinds' data classes; this suite
checks that machinery on every dialect at once. The dialects' own rules are suite DIA, and translation suite TRN.

In [ ]:
import math

from mbse.Expressions import Evaluators as V, Expressions as E, Translators
from mbse.Expressions.Dialects.Ccpp import Evaluators as CV, Expressions as C
from mbse.Expressions.Dialects.Excel import Evaluators as XV, Expressions as X
from mbse.Expressions.Dialects.Latex import Expressions as L
from mbse.Expressions.Dialects.Matlab import Evaluators as MV, Expressions as M
from mbse.Expressions.Dialects.Python import Evaluators as PV, Expressions as P
from mbse.Expressions.Dialects.SystemVerilog import Evaluators as SVV, Expressions as SV
from mbse.Expressions.Framework import Domains as D, Errors as FE, Evaluators as FV, Symbolics as SY, Terms as F, \
    Translators as FT
from mbse.Schemas.Framework import JSON, Proxies, Visitors, YAML
from support import conformance_problems, raises, set_property
from mbse.Expressions import register

store = register(Proxies.OfStore())  # holds every dialect's meta-schemas

DIALECTS = [E.DIALECT, P.DIALECT, M.DIALECT, X.DIALECT, L.DIALECT, C.DIALECT, SV.DIALECT]
EVALUATORS = {E.DIALECT: V.OfAny, P.DIALECT: PV.OfAny, M.DIALECT: MV.OfAny, X.DIALECT: XV.OfAny,
              C.DIALECT: CV.OfAny, SV.DIALECT: SVV.OfAny}  # Latex has none
this = E.variable("this")
basic = E.let_("a", this.age, E.variable("a").ge(18).and_(this.has("email"))).data
samples = {  # one expression per dialect, with every kind
    "Basic": E.operation("and", basic, this.items.all("x", E.variable("x").gt(0))).data,  # a quantifier too
    "Python": P.import_("numpy", P.importfrom("math", "floor", P.let_("a", P.name("this"), P.ifexp(  # every kind
        P.boolop("or", P.boolop("and", P.unaryop("not", P.compare("==", P.attribute(P.name("a"), "z"), 1)),
                                P.unaryop("-", P.attribute(P.name("a"), "w"))), P.binop("*", 2, 3)),
        P.call("np.add", P.attribute(P.name("a"), "x"), P.subscript(P.name("a"), "y")), P.call("f", P.generator("p", P.attribute(P.name("a"), "z"), P.index(P.name("p"), 0), P.name("p"))))),
        alias="f"), alias="np"),
    "Matlab": M.import_("pkg.*", M.binary("&&", M.unary("~", M.field(M.identifier("this"), "age")),
                                          M.call("isfield", "x", M.index(M.arrayfun("p", M.identifier("this"), M.identifier("p")), 1)))),
    "Excel": X.let_("a", X.field(X.name("this"), "age"), X.function("AND", X.infix(">=", X.name("a"), X.cell("A1")),
                                                                         X.prefix("-", X.function("SUM", X.map_("p", X.name("a"), X.infix("*", X.name("p"), 2)))))),
    "Latex": L.where("a", L.member(L.symbol("this"), "age"), L.binary("\\land", L.binary("\\geq", L.symbol("a"), 18), L.binary(
        "\\lor", L.unary("\\lnot", L.function("has", L.symbol("this"), "email")), L.binary(">", L.frac(L.symbol("a"), 2), 1.5)))),
    "Ccpp": C.conditional(C.binary("&&", C.unary("!", C.member(C.identifier("this"), "a")), C.binary(  # every kind
        ">", C.subscript(C.member(C.identifier("this"), "b", "->"), 0), C.constant(1.5, "float"))),
        C.cast("uint8_t", C.call("f", 1)), C.constant(5, "unsigned int")),
    "SystemVerilog": SV.binary("->", SV.inside(SV.member(SV.identifier("this"), "a"), 1, SV.span(2, 5)), SV.conditional(  # every kind
        SV.unary("!", SV.select(SV.identifier("this"), 0)), SV.concatenation(SV.replication(2, SV.vector("10")), SV.range_(SV.identifier("this"), 3, 0)),
        SV.cast("byte", SV.call("$clog2", SV.iterate(SV.method(SV.identifier("this"), "unique"), "sum", "p", SV.identifier("p")))))),
}

## FRM-01 · Every dialect, expression, evaluator and translator implements its protocol

In [ ]:
assert all(isinstance(evaluate, FV.Evaluator) for evaluate in EVALUATORS.values())
for dialect in DIALECTS:
    assert isinstance(dialect, F.Dialect), dialect
    assert conformance_problems(F.Declared, F.Dialect) == []
    for kind in dialect.kinds().values():
        assert conformance_problems(kind, F.Expression) == [], kind
        assert conformance_problems(kind, Visitors.Visitable) == [], kind
        assert conformance_problems(dialect.builders[kind.KIND], Visitors.OfObject) == [], kind
    for expression in F.walk(samples[dialect.name()]):
        assert isinstance(expression, F.Expression) and expression.dialect() is dialect
assert isinstance(V.predicate, FV.Predicate)
assert [d.name() for d in DIALECTS] == ["Basic", "Python", "Matlab", "Excel", "Latex", "Ccpp", "SystemVerilog"] and repr(E.DIALECT) == "<dialect Basic>"
assert list(X.DIALECT.kinds()) == ["constant", "name", "cell", "let", "function", "infix", "prefix", "field", "map"]
for translator in Translators.TRANSLATORS:
    assert isinstance(translator, FT.Translator) and conformance_problems(FT.Pairwise, FT.Translator) == []
for domain in [D.Anything, D.OfTypes("int", int), D.OfValues("odd", lambda v: v % 2 == 1), D.OfUnion(D.Anything)]:
    assert isinstance(domain, D.Domain)
for signature in [D.Function((), D.Anything), D.Same(1, (D.Anything,)), D.Overloaded(D.Function((), D.Anything))]:
    assert isinstance(signature, D.Signature)

## FRM-02 · Forms and `make` are inverses, for every kind of every dialect

In [ ]:
for dialect in DIALECTS:
    kinds = set()
    for expression in F.walk(samples[dialect.name()]):
        form = expression.form()
        kinds.add(form.kind)
        made = dialect.make(form)
        assert type(made) is type(expression) and made.form() == form and made is not expression
    assert kinds == set(dialect.kinds()), (dialect.name(), kinds)  # the samples have every kind
assert E.OfLiteral.resolve(3).form() == F.Form("literal", {"value": 3})
assert E.OfLet.Data("x").form() == F.Form("let", {"name": "x"}, (None, None))  # empty slots are None
assert E.OfOperation.Data().form() == F.Form("operation")
with raises(ValueError, match="Basic has no kind 'constant'"):
    E.DIALECT.make(F.Form("constant"))
with raises(ValueError, match="a literal has no attribute 'name'"):
    E.DIALECT.make(F.Form("literal", {"name": "x"}))
with raises(ValueError, match="a let takes 2 arguments, got 1"):
    E.DIALECT.make(F.Form("let", {"name": "x"}, (basic,)))
with raises(ValueError, match="a call takes at least 1 arguments, got 0"):  # its function, then its arguments
    P.DIALECT.make(F.Form("call"))
made = P.DIALECT.make(F.Form("call", {}, (P.name("f"), P.constant(1), P.constant(2))))
assert made.function.name == "f" and [a.value for a in made.arguments] == [1, 2]
assert E.DIALECT.literal([1]) is None and X.DIALECT.literal(b"x") is None  # no literal kind holds these
assert M.DIALECT.resolve(1).value == 1 and type(M.DIALECT.resolve(True)).KIND == "constant"
with raises(TypeError, match="expected an expression, a native value or a callable taking its builder, got b'x'"):
    X.DIALECT.resolve(b"x")

## FRM-03 · `walk`, `fold` and `same` traverse through forms alone

In [ ]:
shared = E.variable("x")
pair = E.operation("f", shared, shared).data
assert [e.form().kind for e in F.walk(pair)] == ["operation", "variable"]  # shared: visited once
assert [e.form().attributes for e in F.walk(basic)][:3] == [{"name": "a"}, {"name": "get"}, {"name": "this"}]
assert F.fold(pair, lambda term, results: 1 + sum(results)) == 3  # but folded into each use
calls = []
F.fold(pair, lambda term, results: calls.append(term))
assert len(calls) == 2  # once per term
assert F.fold(E.OfLet.Data("x", None, E.OfLiteral.resolve(1)), lambda term, results: results) == [None, []]
loop = E.OfOperation.Builder().name("f").create()
E.OfOperation.Builder(loop).arguments(loop).update()
assert [e for e in F.walk(loop)] == [loop]
with raises(ValueError, match="the expression contains a cycle"):
    F.fold(loop, lambda term, results: None)
other = E.OfOperation.Builder().name("f").create()
E.OfOperation.Builder(other).arguments(other).update()
assert F.same(loop, other) and not F.same(loop, pair)
for a, b, expected in [(1, 1, True), (1, 1.0, False), (math.nan, math.nan, True), (-0.0, 0.0, False),
                       (0.0, 0.0, True), ("x", "x", True), (b"x", b"x", True), (b"x", b"y", False)]:
    assert F.same(E.OfLiteral.resolve(a), E.OfLiteral.resolve(b)) is expected, (a, b)
assert F.same(basic, E.DIALECT.make(basic.form())) and F.same(E.OfLet.Data("x"), E.OfLet.Data("x"))
assert not F.same(E.OfLet.Data("x"), E.let_("x", 1, 2).data) and not F.same(E.OfLiteral.resolve(1), P.constant(1))
assert not F.same(E.operation("f", 1).data, E.operation("f", 1, 2).data)
assert not F.same(E.operation("f").data, E.OfOperation.Data())  # attributes differ

## FRM-04 · Every dialect's meta-schemas and builders are derived from its kinds, and round-trip

In [ ]:
for dialect in DIALECTS[1:]:
    for kind in dialect.kinds().values():
        name = kind.NAME
        assert name.startswith(f"Expressions.{dialect.name()}.Of") and store.schema(name) is kind.Schema and dialect.Builders.schema(name) is kind.Schema
        assert kind.Schema.validate() == [] and dialect.Builders.name_of(kind.Schema) == name
    assert [b.type for b in dialect.Schema.branches] == [k.Schema for k in dialect.kinds().values()]
    expression = samples[dialect.name()]
    schema = dialect.schema_of(expression)
    for encode, decode in [(JSON.ToJSON(store), JSON.FromJSON), (YAML.ToYAML(store), YAML.FromYAML)]:
        back = decode(dialect.Builders).Reachable(schema, encode.Reachable(schema, expression))
        assert F.same(back, expression) and back.validate(bound={"this"}) == []
    assert [b.name for b in dialect.Schema.branches] == list(dialect.kinds())  # named by the kinds' tags
assert list(M.DIALECT.kinds()["binary"].Schema.properties) == ["kind", "operator"]
assert list(X.DIALECT.kinds()["constant"].Schema.properties) == ["kind", "int", "float", "str", "bool"]
assert list(P.DIALECT.kinds()["name"].Schema.adjacencies) == ["used_by"]
assert [k.NAME for k in P.DIALECT.kinds().values()][-6:] == [
    "Expressions.Python.OfUnaryOp", "Expressions.Python.OfIfExp", "Expressions.Python.OfGeneratorExp", "Expressions.Python.OfLet",
    "Expressions.Python.OfImport", "Expressions.Python.OfImportFrom"]
with raises(TypeError, match="not an expression: 3"):
    E.DIALECT.schema_of(3)
with raises(TypeError, match="a constant cannot hold a bytes"):
    JSON.ToJSON(store).Reachable(X.DIALECT.kinds()["constant"].Schema, X.DIALECT.kinds()["constant"](b"x"))

Builder = X.DIALECT.builders["infix"]
infix = Builder().set("operator", "+").argument("left", 1).argument("right", 2).argument("left", 3).create()
assert infix.left.value == 3 and infix.right.value == 2 and X.DIALECT.builders["constant"]().set("value", 1).create().value == 1
call = P.DIALECT.builders["call"]().arguments(1, lambda a: a.select(2)).argument("function", P.name("f")).create()
assert call.function.name == "f" and [a.value for a in call.arguments] == [1, 2]  # a slot, then variadic arguments
again = P.DIALECT.builders["call"](call).arguments(3).clone()
assert again.function is call.function and [a.value for a in again.arguments] == [1, 2, 3]
with raises(KeyError, match="an infix has no attribute 'name'"):
    Builder().set("name", "x")
with raises(TypeError, match="an infix has no variadic arguments"):
    Builder().arguments(1)
with raises(KeyError, match="an infix has no argument 'operand'"):
    Builder().argument("operand", 1)


def add(builder, index, target):
    """Adds an `arguments` entry through the protocol, as FromPlain does."""
    def fill(entry):
        entry.link("argument", lambda k: k.set(target))
        if index is not None:
            set_property(entry, "index", index)
    builder.adjacency("arguments", lambda a: a.add(fill))
    return builder


with raises(ValueError, match="an infix's left is argument 0 and its right argument 1, got index 2"):
    add(Builder().set("operator", "+"), 2, infix).create()
with raises(ValueError, match="a field's value is argument 0, got index None"):
    add(M.DIALECT.builders["field"]().set("name", "x"), None, M.constant(1)).create()
with raises(TypeError, match="an argument must be an expression, got _Infix"):  # an expression of its own dialect
    add(M.DIALECT.builders["field"]().set("name", "x"), 0, infix).create()


class Three(F.Term):  # a kind with three slots, to check the message
    KIND, SLOTS = "if", ("condition", "then", "otherwise")


assert F._slots_message(Three, 5) == "an if's condition is argument 0, its then argument 1 and its otherwise argument 2, got index 5"

## FRM-05 · Validation follows each kind's role, in every dialect

In [ ]:
assert [samples[d.name()].validate(bound={"this"}, core=True) for d in DIALECTS] == [[], [], [], [], [], [], []]
assert P.subscript(P.name("t"), "x").validate() == ["value: name 't' is not bound"]
assert P.call("f", 1).validate() == ["function: name 'f' is not bound"]  # a slot, then variadic arguments
assert P.call(P.name("f"), P.name("x")).validate(bound={"f"}) == ["argument 0: name 'x' is not bound"]
assert P.import_("numpy", P.name("np"), alias="np").validate() == []  # an import binds a name; its alias is optional
assert P.import_("numpy", P.name("np")).validate() == ["body: name 'np' is not bound"]
assert P.DIALECT.kinds()["import"]("numpy").validate() == ["an import needs a body"]
assert P.name("not a name").validate() == ["a name must be an identifier, got 'not a name'"]  # a kind's own check
assert X.cell("A1").validate() == [] and X.cell("B2", "Sheet1").validate(core=True) == []  # cells need no binding
assert M.DIALECT.kinds()["field"]().validate() == ["a field needs a name", "a field needs a value"]
assert M.DIALECT.kinds()["binary"]("&&", M.constant(1)).validate() == ["a binary needs a right"]
assert M.binary("**", 1, 2).validate() == [] and M.binary("**", 1, 2).validate(core=True) == ["'**' is not a core operation"]
assert M.field(M.identifier("s"), "any name").validate(bound={"s"}, core=True) == []  # a field's vocabulary is open
assert X.function("NOT", True, False).validate() == ["NOT takes 1 arguments, got 2"] and X.function("AND", True).validate() == []  # AND: any number
assert X.let_("a", X.name("a"), X.name("a")).validate() == ["value: name 'a' is not bound"]
assert X.DIALECT.kinds()["constant"](b"x").validate() == ["a constant cannot hold a bytes"]
assert X.DIALECT.kinds()["constant"]([]).validate() == ["a constant must hold a native value, got list"]
assert M.DIALECT.kinds()["call"](7).validate() == ["a call's function must be a str, got int"]
assert E.OfOperation.Data("add", (1,)).validate() == [
    "add takes 2 arguments, got 1", "argument 0: not an expression: 1"]

## FRM-06 · Domains and signatures

In [ ]:
Int, Float, Str, Odd = D.OfTypes("int", int), D.OfTypes("float", float), D.OfTypes("str", str), D.OfValues("odd", lambda v: v % 2 == 1)
Number = D.OfUnion(Int, Float)
assert Int.contains(1) and not Int.contains(True) and not Int.contains(1.0) and repr(Int) == "int"
assert D.OfTypes("bool", bool).contains(True) and not D.OfTypes("bool", bool).contains(1)
assert Str.contains("a") and not Str.contains(b"a")
assert Number.name() == "int | float" and repr(Number) == "int | float" and Number.contains(1.5) and not Number.contains("1")
assert Number.includes(Int) and Number.includes(D.OfUnion(Float, Int)) and not Int.includes(Number)
assert D.OfTypes("num", int, float).includes(Number) and not Number.includes(Str) and not Number.includes(D.Anything)
assert Odd.contains(3) and Odd.includes(Odd) and not Odd.includes(Int) and Odd.includes(D.OfUnion(Odd)) and repr(Odd) == "odd"
assert D.Anything.includes(Number) and D.Anything.contains(None) and repr(D.Anything) == "any" and D.Anything.name() == "any"
assert D.overlaps(Int, D.Anything) and D.overlaps(Number, Int) and not D.overlaps(Int, Str)
f = D.Function((Int, Str), Float)
assert f.arity() == 2 and f.result([Int, Str]) is Float and f.result([D.Anything, Str]) is Float
assert f.result([Str, Str]) is None and f.result([Int]) is None and f.describe() == "(int, str) -> float"
assert f.exact([Int, Str]) and not f.exact([D.Anything, Str])
same = D.Same(2, (Int, Float))
assert same.result([Int, Int]) is Int and same.result([Int, Float]) is None and same.result([Int]) is None
assert same.result([D.Anything, D.Anything]).name() == "int | float" and same.result([D.Anything, Float]) is Float
assert same.describe() == "(T, T) -> T for T in int | float" and D.Same(1, (Int,), Str).describe() == "(T) -> str for T in int"
assert D.Same(2, (Int, Float), Str).result([D.Anything, D.Anything]) is Str  # one result, however many match
assert D.Opaque().arity() == -1 and D.Opaque(Int).result([Str, Str, Str]) is Int and D.Opaque().describe() == "(...) -> any"
either = D.Either(2)
assert either.arity() == 2 and either.result([Int, Int]) is Int and either.result([Int, Str]).name() == "int | str"
assert either.result([Int]) is None and either.describe() == "(T, T) -> T"
assert same.exact([Int, Int]) and not same.exact([Int, Float]) and not same.exact([Int])
over = D.Overloaded(D.Same(2, (Int,)), D.Function((Number, Number), Float))
assert over.arity() == 2 and over.result([Int, Int]) is Int and over.result([Int, Float]) is Float
assert over.result([D.Anything, Int]).name() == "int | float" and over.result([Str, Str]) is None
assert over.describe() == "(T, T) -> T for T in int | (int | float, int | float) -> float"

## FRM-07 · `infer` finds an expression's domain from its vocabulary's signatures

In [ ]:
from mbse.Expressions.Dialects.Basic import Domains as BD

assert E.DIALECT.infer(basic, {"this": BD.Object}) is BD.Bool
assert E.DIALECT.infer(E.literal(1).add(2)) is BD.Int and E.DIALECT.infer(E.literal(1.5).neg()) is BD.Float
assert E.DIALECT.infer(this.age, {"this": BD.Object}) is D.Anything
assert E.DIALECT.infer(this.age.add(1), {"this": BD.Object}) is BD.Int  # only int + int applies
assert E.DIALECT.infer(E.operation("contains", 1, 2)) is D.Anything  # an extension
assert E.DIALECT.infer(E.let_("a", 1, E.let_("b", "s", E.variable("a")))) is BD.Int
with raises(TypeError, match="add cannot take (int, float); it takes (T, T) -> T for T in int | float"):
    E.DIALECT.infer(E.literal(1).add(1.5))
with raises(TypeError, match="has cannot take (int, str); it takes (object | record, str) -> bool"):
    E.DIALECT.infer(E.literal(1).has("x"))
with raises(ValueError, match="cannot infer the domain of an invalid expression: variable 'this' is not bound"):
    E.DIALECT.infer(this)
assert BD.of(E.OfLiteral.resolve(1)) is BD.Object and BD.of(b"") is BD.Bytes
with raises(TypeError, match="a list is not a value of the Basic dialect"):
    BD.of([])
assert M.DIALECT.infer(samples["Matlab"], {"this": D.Anything}).name() == "logical"
assert X.DIALECT.infer(samples["Excel"], {"this": D.Anything}).name() == "logical"
assert L.DIALECT.infer(samples["Latex"], {"this": D.Anything}).name() == "truth"
assert P.DIALECT.infer(samples["Python"], {"this": D.Anything}) is D.Anything  # a call gives what is not known
assert P.DIALECT.infer(P.boolop("and", 1, "a")).name() == "int | str"  # Python's and gives an operand
assert P.DIALECT.infer(P.import_("math", P.binop("+", 1, True))).name() == "int"

## FRM-08 · The interpreter evaluates by role and raises what validation reports

In [ ]:
interpreter = FV.Interpreter(M.DIALECT, {"binary": {}, "unary": {}, "call": {}, "field": lambda a, n, s: None})
with raises(TypeError, match="not an expression: 3"):
    interpreter(3)
with raises(ValueError, match="a constant needs a value"):
    interpreter(M.constant(None))
with raises(KeyError, match="identifier 'x' is not bound"):
    interpreter(M.identifier("x"))
with raises(NotImplementedError, match="'&&' is not a core operation"):
    interpreter(M.binary("&&", 1, 2))
assert interpreter(M.field(M.identifier("s"), "f"), {"s": 1}) is None
with raises(TypeError, match="isfield takes 2 arguments, got 1"):
    FV.Interpreter(M.DIALECT, {"call": {"isfield": None}})(M.call("isfield", 1))
cycle = X.DIALECT.kinds()["let"]("a", X.constant(1))
cycle.body = cycle
with raises(ValueError, match="the expression contains a cycle"):
    XV.OfAny(cycle)
with raises(ValueError, match="a let needs a body"):
    XV.OfAny(X.DIALECT.kinds()["let"]("a", X.constant(1)))
with raises(ValueError, match="an import needs a body"):
    MV.OfAny(M.DIALECT.kinds()["import"]("pkg.*"))

scope = SY.Variables({"x": 1})
assert isinstance(scope, SY.Scope) and scope.lookup(E.variable("x").data) == 1
inner = scope.bind("x", 2)
assert inner.lookup(E.variable("x").data) == 2 and scope.lookup(E.variable("x").data) == 1  # bind makes a new scope
with raises(KeyError, match="variable 'y' is not bound"):
    scope.lookup(E.variable("y").data)
with raises(NotImplementedError, match="Variables has no imports"):
    scope.enter(M.import_("pkg.*", 1))
assert V.OfAny(E.variable("x"), inner) == 2  # an evaluator takes a scope as well as a mapping
with raises(NotImplementedError, match="'dist' is not a core operation"):  # no extension: the vocabulary only
    interpreter(M.call("dist", 1, 2))

## FRM-09 · Symbolics: the names an expression needs, and the imports it declares

In [ ]:
assert SY.free(basic) == {"this"} and SY.free(E.literal(1)) == set()
assert SY.free(E.let_("a", E.variable("a"), E.variable("a"))) == {"a"}  # the name is bound in the body only
assert SY.free(samples["Python"]) == {"this"}  # np, f and a are bound by imports and the let
assert SY.free(P.call("hasattr", P.name("x"), "k")) == {"x"}  # builtins are ambient
assert SY.free(X.infix("+", X.cell("A1"), X.name("r"))) == {"r"}  # cells are resolved by the workbook, not bound
assert SY.free(M.import_("pkg.*", M.identifier("x"))) == {"x"}  # MATLAB's imports bind functions, not variables
knot = E.OfOperation.Builder().name("f").create()
E.OfOperation.Builder(knot).arguments(knot, E.variable("x")).update()
assert SY.free(knot) == {"x"}  # cycles end the search
assert SY.free(E.OfLet.Data("a", None, E.variable("b").data)) == {"b"}  # empty slots are skipped
assert [type(i).KIND for i in SY.imports(samples["Python"])] == ["import", "importfrom"] and SY.imports(basic) == []
assert [i.name for i in SY.imports(samples["Matlab"])] == ["pkg.*"]
assert FE.NameError is NameError and FE.ImportError is ImportError  # Python's own exceptions, by the framework's names
assert FE.OverflowError is OverflowError and FE.ZeroDivisionError is ZeroDivisionError

## FRM-10 · A dialect may extend another: its kinds are the base's and its own

In [ ]:
import dataclasses
from typing import Any

from mbse.Expressions import Domains as BD


@dataclasses.dataclass(eq=False)
class _Twice(F.Term):
    """`twice(x)`, x + x: the one kind of an extension of Basic."""

    KIND = "twice"
    ROLE = F.APPLICATION
    SLOTS = ("operand",)
    operand: Any = None


Twice = F.Declared("Twice", [_Twice], domain_of=BD.of, extends=E.DIALECT)
x = E.variable("x")
mixed = x.add(_Twice(x.data)).data  # a writer of the base holds the extension's term
assert isinstance(mixed.arguments[1], _Twice) and _Twice in E.DIALECT.terms() and Twice in E.DIALECT.extensions
assert Twice.base is E.DIALECT and Twice.classes == (*E.DIALECT.classes, _Twice) and E.DIALECT.base is None
assert list(Twice.kinds()) == [*E.DIALECT.kinds(), "twice"] and Twice.terms() == Twice.classes
assert Twice.validate(mixed, bound={"x"}) == []
problems = E.DIALECT.validate(mixed, bound={"x"})  # the base cannot validate what holds the extension's terms
assert len(problems) == 1 and problems[0].startswith("argument 1: not an expression: ")
interpreter = FV.Interpreter(Twice, {"operation": V.OPERATIONS, "quantifier": V.QUANTIFIERS,
                                     "twice": lambda thunks, node, scope: thunks[0]() * 2})
assert interpreter(mixed, {"x": 3}) == 9
assert "Expressions.Twice.OfTwice" in Twice.schemas and "Expressions.OfOperation" in Twice.schemas
assert [b.name for b in Twice.Schema.branches] == [*E.DIALECT.kinds(), "twice"]
text = JSON.ToJSON(Twice.Builders).Reachable(Twice.schema_of(mixed), mixed)
assert F.same(JSON.FromJSON(Twice.Builders).Reachable(Twice.schema_of(mixed), text), mixed)  # the base reads it too


@dataclasses.dataclass(eq=False)
class _Literal(F.Term):
    KIND = "literal"
    ROLE = F.APPLICATION


with raises(ValueError, match="Clash would declare 'literal', which Basic has"):
    F.Declared("Clash", [_Literal], domain_of=BD.of, extends=E.DIALECT)